# Proyecto #1: Biodiversity at Scale
## Parte 6: Transfer Learning y Fine-Tuning
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
Evaluar y comparar rigurosamente tres escenarios con un backbone convolucional preentrenado (`ResNet-50` o `ResNet-18` sobre ImageNet):
1. **Experimento A: Feature Extraction**: Backbone congelado; solo se entrena la cabeza de clasificación lineal.
2. **Experimento B: Partial Fine-Tuning**: Descongelamiento de las capas finales (`layer4`) con learning rate moderado ($10^{-4}$).
3. **Experimento C: Full Fine-Tuning**: Descongelamiento total de parámetros con learning rate diferencial bajo ($10^{-5}$ en backbone, $10^{-3}$ en cabeza).
4. Registrar los experimentos **E7, E8 y E9** en el Tracker oficial.


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything
from src.data.dataset import SyntheticINatDataset
from src.data.dataloader import build_dataloaders
from src.models.factory import build_model, count_parameters
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.trainer import Trainer
from src.utils.tracking import ExperimentTracker

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))


### 1. Preparación del Pipeline de Datos (Resolución $224 \times 224$)


In [ ]:
N_CLASSES = 50
BATCH_SIZE = 32
EPOCHS = 4
IMG_SIZE = 224

train_ds = SyntheticINatDataset(num_samples=50 * 35, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED)
val_ds = SyntheticINatDataset(num_samples=50 * 10, num_classes=N_CLASSES, img_size=IMG_SIZE, seed=SEED+1)
train_loader, val_loader = build_dataloaders(train_ds, val_ds, batch_size=BATCH_SIZE, num_workers=2, seed=SEED)
criterion = build_criterion("cross_entropy")


### 2. Experimento A: Feature Extraction (E7)


In [ ]:
model_fe = build_model("resnet18", num_classes=N_CLASSES, mode="feature_extraction", pretrained=True)
total_p, train_p, total_m, train_m = count_parameters(model_fe)
print(f"Feature Extraction - Parámetros entrenables: {train_p:,} de {total_p:,}")

opt_fe = build_optimizer(model_fe, opt_type="adamw", lr=1e-3)
trainer_fe = Trainer(model_fe, criterion, opt_fe, device=device, use_amp=True)
hist_fe, best_fe, vram_fe, time_fe = trainer_fe.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

tracker.log_experiment("E7", "ResNet-18 (Pretrained)", "AdamW", "BN+Dropout", "Standard", "Frozen Backbone", "No", best_fe, time_fe, vram_fe, total_m)


### 3. Experimento B: Partial Fine-Tuning (E8)


In [ ]:
model_pft = build_model("resnet18", num_classes=N_CLASSES, mode="partial_fine_tuning", pretrained=True)
param_groups = model_pft.get_parameter_groups(backbone_lr=1e-4, head_lr=1e-3)
opt_pft = build_optimizer(model_pft, opt_type="adamw", param_groups=param_groups)

trainer_pft = Trainer(model_pft, criterion, opt_pft, device=device, use_amp=True)
hist_pft, best_pft, vram_pft, time_pft = trainer_pft.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

tracker.log_experiment("E8", "ResNet-18 (Pretrained)", "AdamW (Diff LR)", "BN+Dropout", "Standard", "Partial FT (layer4)", "No", best_pft, time_pft, vram_pft, total_m)


### 4. Experimento C: Full Fine-Tuning (E9)


In [ ]:
model_fft = build_model("resnet18", num_classes=N_CLASSES, mode="full_fine_tuning", pretrained=True)
param_groups_full = model_fft.get_parameter_groups(backbone_lr=1e-5, head_lr=1e-3)
opt_fft = build_optimizer(model_fft, opt_type="adamw", param_groups=param_groups_full)

trainer_fft = Trainer(model_fft, criterion, opt_fft, device=device, use_amp=True)
hist_fft, best_fft, vram_fft, time_fft = trainer_fft.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)

tracker.log_experiment("E9", "ResNet-18 (Pretrained)", "AdamW (Diff LR)", "BN+Dropout", "Standard", "Full Fine-Tuning", "No", best_fft, time_fft, vram_fft, total_m)
